# CSD Key Selection: Share vs Frequency

This notebook analyzes frequency distributions for two CSD key-selection rules:

- Fixed share rule: `share(d,t) >= 0.5`
- Above-uniform rule: `share(d,t) > 1 / K(d)`, where `K(d)` is the number of distinct replacement tokens observed for draft token `d`

The goal is to check whether a rule mainly keeps rare pairs and how much frequency mass it covers.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "benchmark" / "csd").exists():
    REPO_ROOT = Path("/root/sglang")

TABLE_PATH = Path(os.environ.get(
    "CSD_TABLE_PATH",
    REPO_ROOT / "benchmark/csd/runs/redpajama/csd_table_redpajama_logits_gated_6domains_n1000_Qwen3.5-35B-A3B_mtp_EAGLE_steps3_topk1_draft3_temp1.0_ratio0.3.json",
))
TABLE_PATH

In [ ]:
with TABLE_PATH.open() as f:
    payload = json.load(f)

entries = payload["entries"]
lhs_total = Counter()
lhs_distinct = Counter()

for row in entries:
    lhs = row["lhs_token"]
    freq = int(row["freq"])
    lhs_total[lhs] += freq
    lhs_distinct[lhs] += 1

rows = []
for row in entries:
    lhs = row["lhs_token"]
    rhs = row["rhs_token"]
    freq = int(row["freq"])
    total = lhs_total[lhs]
    k = lhs_distinct[lhs]
    share = freq / total
    uniform_share = 1 / k
    rows.append({
        "key": row["key"],
        "lhs_token": lhs,
        "rhs_token": rhs,
        "freq": freq,
        "lhs_total": total,
        "num_replacements": k,
        "share": share,
        "uniform_share": uniform_share,
        "above_uniform": share > uniform_share,
        "above_or_equal_uniform": share >= uniform_share,
        "score_count_squared_over_total": freq * freq / total,
    })

df = pd.DataFrame(rows)
print(f"entries={len(df):,}")
print(f"lhs_tokens={df['lhs_token'].nunique():,}")
print(f"total_count_mass={df['freq'].sum():,}")
df.head()

## Helpers

`entry_ratio` answers how many hash keys survive. `count_mass_ratio` answers how much observed pair frequency mass survives. The second one is useful because a rule can keep few entries but still keep important high-frequency pairs.

In [ ]:
FREQ_BINS = [0, 1, 2, 5, 10, 50, float("inf")]
FREQ_LABELS = ["freq=1", "freq=2", "freq=3-5", "freq=6-10", "freq=11-50", "freq>50"]

def with_freq_bucket(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["freq_bucket"] = pd.cut(out["freq"], bins=FREQ_BINS, labels=FREQ_LABELS, right=True)
    return out

def summarize_mask(name: str, mask: pd.Series) -> dict:
    selected = df[mask]
    total_entries = len(df)
    total_mass = df["freq"].sum()
    selected_entries = len(selected)
    selected_mass = selected["freq"].sum()
    return {
        "rule": name,
        "selected_entries": selected_entries,
        "entry_ratio": selected_entries / total_entries,
        "selected_count_mass": selected_mass,
        "count_mass_ratio": selected_mass / total_mass,
        "median_freq": selected["freq"].median() if selected_entries else 0,
        "mean_freq": selected["freq"].mean() if selected_entries else 0,
        "freq1_entries": int((selected["freq"] == 1).sum()),
        "freq1_ratio_in_selected": float((selected["freq"] == 1).mean()) if selected_entries else 0,
    }

def bucket_distribution(name: str, mask: pd.Series) -> pd.DataFrame:
    selected = with_freq_bucket(df[mask])
    grouped = selected.groupby("freq_bucket", observed=False).agg(
        entries=("freq", "size"),
        count_mass=("freq", "sum"),
    ).reindex(FREQ_LABELS)
    grouped["rule"] = name
    grouped["entry_pct_in_rule"] = grouped["entries"] / max(len(selected), 1)
    grouped["count_mass_pct_in_rule"] = grouped["count_mass"] / max(selected["freq"].sum(), 1)
    return grouped.reset_index().rename(columns={"index": "freq_bucket"})

## Overall Frequency Distribution

This shows how sparse the raw calibration table is before any key-selection rule.

In [ ]:
overall = bucket_distribution("all_entries", pd.Series(True, index=df.index))
overall

## Compare Rules

We compare fixed `share >= 0.5` against the adaptive baseline `share > 1/K`. The adaptive rule asks whether a replacement is more frequent than uniform among the observed replacements for that draft token.

In [ ]:
rules = []
for min_count in [1, 2, 3, 4, 5, 6, 10]:
    rules.append((f"share>=0.5 & freq>={min_count}", (df["share"] >= 0.5) & (df["freq"] >= min_count)))
    rules.append((f"share>1/K & freq>={min_count}", df["above_uniform"] & (df["freq"] >= min_count)))

summary = pd.DataFrame([summarize_mask(name, mask) for name, mask in rules])
summary

In [ ]:
dist = pd.concat([bucket_distribution(name, mask) for name, mask in rules], ignore_index=True)
dist.head(20)

## Focused Tables

The next two tables show the frequency distribution for each rule. `entry_pct_in_rule` is the percentage of selected keys in the bucket. `count_mass_pct_in_rule` is the percentage of selected observed frequency mass in the bucket.

In [ ]:
share05_dist = dist[dist["rule"].str.startswith("share>=0.5")].copy()
share05_dist.pivot_table(
    index="rule",
    columns="freq_bucket",
    values="entries",
    aggfunc="sum",
    fill_value=0,
).loc[[f"share>=0.5 & freq>={m}" for m in [1,2,3,4,5,6,10]]]

In [ ]:
above_uniform_dist = dist[dist["rule"].str.startswith("share>1/K")].copy()
above_uniform_dist.pivot_table(
    index="rule",
    columns="freq_bucket",
    values="entries",
    aggfunc="sum",
    fill_value=0,
).loc[[f"share>1/K & freq>={m}" for m in [1,2,3,4,5,6,10]]]

## Plots

The left plot is selected-key count by frequency bucket. The right plot is selected frequency mass by bucket.

In [ ]:
plot_rules = [
    "share>=0.5 & freq>=1",
    "share>=0.5 & freq>=2",
    "share>=0.5 & freq>=3",
    "share>1/K & freq>=1",
    "share>1/K & freq>=2",
    "share>1/K & freq>=3",
    "share>1/K & freq>=5",
]

plot_dist = dist[dist["rule"].isin(plot_rules)].copy()

entry_pivot = plot_dist.pivot_table(index="rule", columns="freq_bucket", values="entries", fill_value=0)
mass_pivot = plot_dist.pivot_table(index="rule", columns="freq_bucket", values="count_mass", fill_value=0)
entry_pivot = entry_pivot.loc[plot_rules]
mass_pivot = mass_pivot.loc[plot_rules]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
entry_pivot.plot(kind="bar", stacked=True, ax=axes[0])
axes[0].set_title("Selected Entries by Frequency Bucket")
axes[0].set_ylabel("entries")
axes[0].tick_params(axis="x", rotation=45)

mass_pivot.plot(kind="bar", stacked=True, ax=axes[1])
axes[1].set_title("Selected Count Mass by Frequency Bucket")
axes[1].set_ylabel("sum(freq)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()

## Why `share >= 0.5` Can Be Too Harsh

For draft tokens with many possible replacements, a useful replacement can be clearly above the uniform baseline while still far below 0.5. The next cell shows how selected pairs are distributed by `K` under the adaptive rule.

In [ ]:
def k_bucket(k: int) -> str:
    if k == 1:
        return "K=1"
    if k == 2:
        return "K=2"
    if k == 3:
        return "K=3"
    if k <= 5:
        return "K=4-5"
    if k <= 10:
        return "K=6-10"
    return "K>10"

tmp = df[df["above_uniform"]].copy()
tmp["k_bucket"] = tmp["num_replacements"].map(k_bucket)
tmp.groupby("k_bucket").agg(
    entries=("freq", "size"),
    count_mass=("freq", "sum"),
    median_share=("share", "median"),
    median_uniform_share=("uniform_share", "median"),
).reindex(["K=1", "K=2", "K=3", "K=4-5", "K=6-10", "K>10"])

## Inspect Examples

These are high-frequency pairs that pass `share > 1/K` but fail `share >= 0.5`. They are the kind of pairs a fixed 0.5 cutoff would throw away.

In [ ]:
interesting = df[(df["above_uniform"]) & (df["share"] < 0.5)].sort_values(
    ["freq", "share"], ascending=[False, False]
)
interesting[[
    "lhs_token", "rhs_token", "freq", "lhs_total", "num_replacements", "share", "uniform_share", "score_count_squared_over_total"
]].head(30)